# Discovered vs Known Translation Generators on Heat and Burgers

This notebook compares fitted translation generators against the known canonical translation family on both stable PDEs.

The notebook uses:

- `fit_translation_generator(...)`
- `compare_generator_spans(...)`
- `render_generator_family(...)`
- `verify_translation_generator(...)`

The goal is to see where the current fitting path is robust versus where it only behaves cleanly on the easiest synthetic slice.


In [ ]:
import numpy as np

from pdelie import GeneratorFamily
from pdelie.data import generate_burgers_1d_field_batch, generate_heat_1d_field_batch
from pdelie.residuals import BurgersResidualEvaluator, HeatResidualEvaluator
from pdelie.symmetry import compare_generator_spans, fit_translation_generator, render_generator_family
from pdelie.verification import verify_translation_generator


In [ ]:
def make_known_translation_generator() -> GeneratorFamily:
    basis_spec = {
        "variables": ["t", "x", "u"],
        "component_names": ["xi"],
        "basis_terms": [
            {"label": "1", "powers": [0, 0, 0]},
            {"label": "t", "powers": [1, 0, 0]},
            {"label": "x", "powers": [0, 1, 0]},
            {"label": "u", "powers": [0, 0, 1]},
        ],
        "component_ordering": ["xi"],
        "term_ordering": ["1", "t", "x", "u"],
        "layout": "component_major",
    }
    return GeneratorFamily(
        parameterization="polynomial_translation_affine",
        coefficients=np.array([[1.0, 0.0, 0.0, 0.0]], dtype=float),
        basis_spec=basis_spec,
        normalization="l2_unit",
        diagnostics={},
    )


known = make_known_translation_generator()


In [ ]:
pde_specs = [
    ("heat", generate_heat_1d_field_batch, HeatResidualEvaluator()),
    ("burgers", generate_burgers_1d_field_batch, BurgersResidualEvaluator()),
]

rows = []
representatives = {}

for pde_name, generator_fn, residual_evaluator in pde_specs:
    for seed in [400, 401, 402]:
        training = generator_fn(batch_size=4, num_times=33, num_points=64, seed=seed)
        heldout = generator_fn(batch_size=3, num_times=33, num_points=64, seed=seed + 1000)
        discovered = fit_translation_generator(training, residual_evaluator)
        span_report = compare_generator_spans(known, discovered)
        verification = verify_translation_generator(heldout, discovered, residual_evaluator)
        rows.append(
            {
                "pde": pde_name,
                "seed": seed,
                "principal_angle": float(span_report["principal_angles_radians"][0]),
                "projection_residual": span_report["projection_residual"]["summary"],
                "verification_classification": verification.classification,
                "fit_mode": discovered.diagnostics["fit_mode"],
                "svd_span_distance": discovered.diagnostics["svd_span_distance"],
            }
        )
        representatives.setdefault(pde_name, discovered)

rows


## Inspect representative discovered vs known generators


In [ ]:
for pde_name, discovered in representatives.items():
    print(f"=== {pde_name} ===")
    print("known:")
    for line in render_generator_family(known):
        print("  ", line)
    print("discovered:")
    for line in render_generator_family(discovered):
        print("  ", line)
    print("diagnostics:")
    print(discovered.diagnostics)
    print()


## Quick takeaways

Look at the principal angles, projection residuals, verification classifications, and fallback usage. If a PDE repeatedly needs the reference fallback or produces visibly less stable rendered generators, that is a sign the current fitting path is more fragile there.
